# CS383: Data Science and Machine Learning
## Lecture 6 — Feature Engineering

### Where this fits

Lecture 5 was about *understanding* NYC 311 — its center, spread, shape, and relationships. Every model
starting Week 7 needs that same data turned into clean, comparable numbers. This lecture is the practical
bridge between the two: one concept at a time, starting simple on 311 (already-familiar territory, no new
dataset to get oriented in), then a second, more hands-on pass on the restaurant inspections data from
Assignment 1 — pulling several of these tools together at once, the way a real project actually requires.

---

### Before we open the notebook: an unplugged warm-up

No coding for this part. First, you'll sort a handful of everyday categories (borough, T-shirt size, and
more) into nominal vs. ordinal and pick the matching transform. Then, using just two numbers everyone
already has a feel for — age and income — plus two more real-world pairs, you'll guess who's "most
similar" to a target, and see what raw numbers say versus what happens once features are put on
comparable scale. Same ideas Parts 2 and 3 below cover in code, just felt out by hand first.

**[Open the "Getting Your Features Ready" Activity](https://thitimas.github.io/cs383-fa26-materials-public/lect06/scaling_unplugged_activity.html)**

Takes about 15-20 minutes. Come back here once you've been through all the rounds.

---

## Part 1 — Why Feature Engineering

Every model we build starting next week expects its input as **numbers, on comparable scales**. Real
data almost never arrives that way — it arrives as category strings, wildly different numeric ranges, and
columns that were never designed with a model in mind. Feature engineering is the work of turning what you
collected into what a model can actually use, without quietly breaking anything along the way.

### Setup — NYC 311

Same dataset and live-pull-with-fallback pattern from Lectures 1, 2, 3, and 5.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

try:
    raw_path = os.path.expanduser("~/shared/nyc311_snapshot.csv")
    complaints_df = pd.read_csv(raw_path).head(8000)
    complaints_df["created_date"] = pd.to_datetime(complaints_df["created_date"])
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = True

except Exception:
    rng = np.random.default_rng(383)
    n = 8000
    complaint_types = ["Noise - Residential", "Illegal Parking", "HEAT/HOT WATER",
                        "Blocked Driveway", "Street Condition", "Water System",
                        "PAINT/PLASTER", "Damaged Tree", "Sewer", "Rodent"]
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]

    n_days = 90
    days = pd.date_range("2026-01-01", periods=n_days, freq="D")
    day_weight = np.where(days.dayofweek >= 5, 0.6, 1.0)
    day_weight = day_weight / day_weight.sum()

    day_idx = rng.choice(n_days, size=n, p=day_weight)
    created = days[day_idx] + pd.to_timedelta(rng.integers(0, 24 * 60, size=n), unit="m")

    still_open = rng.random(n) < 0.15
    resolution_hours = rng.gamma(shape=2.0, scale=20.0, size=n)
    closed = created + pd.to_timedelta(resolution_hours, unit="h")

    complaints_df = pd.DataFrame({
        "complaint_type": rng.choice(
            complaint_types, size=n,
            p=[0.18, 0.15, 0.14, 0.10, 0.10, 0.09, 0.08, 0.06, 0.05, 0.05],
        ),
        "borough": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "created_date": created,
        "closed_date": np.where(still_open, pd.NaT, closed),
    })
    complaints_df["closed_date"] = pd.to_datetime(complaints_df["closed_date"])
    live = False

complaints_df["resolution_time_hours"] = (
    complaints_df["closed_date"] - complaints_df["created_date"]
).dt.total_seconds() / 3600
complaints_df["hour_filed"] = complaints_df["created_date"].dt.hour

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(complaints_df):,} 311 records")
complaints_df[["complaint_type", "borough", "hour_filed", "resolution_time_hours"]].head()

In [ ]:
complaints_df.__________

Look at those dtypes: `complaint_type` and `borough` are `object` (text). `hour_filed` and
`resolution_time_hours` are numeric, but on very different scales — `hour_filed` only ever runs 0-23,
while `resolution_time_hours` can run into the hundreds.

Neither of these is a problem for a human reading the table. Both are a problem for a model:

- Regression and k-NN (Weeks 7-8) do arithmetic directly on your feature values — text can't be
  subtracted or multiplied, so it has to become numbers first.
- Even once everything is numeric, a feature with a much bigger raw range can end up dominating a
  model's calculations, not because it's more important, but simply because its numbers are bigger.

### Quick recap from Lecture 1: nominal vs. ordinal

Which encoding strategy makes sense depends on the same distinction from Lecture 1's Types of Data:

- `complaint_type` and `borough` are **nominal** — categories with no natural order. Queens isn't
  "more" or "less" than Brooklyn.

311 doesn't hand us a clean ordinal column today — you'll meet a genuine one (restaurant `grade`: A
better than B better than C) later in this lecture, once restaurant data comes back in.

---

## Part 2 — Encoding Categorical Variables

### One-hot encoding, for nominal categories

**One-hot encoding** turns one categorical column into several binary (0/1) columns, one per category —
a 1 marks which category that row belongs to, everywhere else is 0.

In [ ]:
pd.__________(complaints_df[["borough"]], drop_first=False).head()

Each borough became its own column, filled with `True`/`False` (equivalent to 1/0). A row filed in
Brooklyn has `borough_BROOKLYN = True` and every other borough column `False`.

`drop_first=False` keeps all five columns. In practice you'll often see `drop_first=True`, which drops
one category's column entirely (it becomes the case where every other column reads 0) — this avoids
redundant, perfectly-correlated columns for algorithms sensitive to that (linear regression among them).
Either choice is defensible; just be consistent and know which one you picked.

### The same idea, with scikit-learn's `OneHotEncoder`

`pd.get_dummies()` is convenient for a quick look, but scikit-learn's `OneHotEncoder` is what you'll
actually use in a model pipeline — it remembers the categories it was fit on, and can be told exactly
what to do if a brand new, never-before-seen category shows up in test data.

In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
borough_encoded = encoder.__________(complaints_df[["borough"]])

pd.DataFrame(borough_encoded, columns=encoder.get_feature_names_out(["borough"])).head()

`handle_unknown="ignore"` matters more than it looks: if your test set (or a brand new prediction
request) contains a category the encoder never saw during `fit()` — a borough or cuisine that never
showed up in training — this setting tells it to encode that row as all zeros instead of crashing.
Without it, a single unseen category can take down your entire prediction pipeline.

### Ordinal encoding, for categories with a real order

311 doesn't give us a clean ordinal column to encode today, so here's the idea in miniature on a tiny
made-up example first — you'll apply the same tool for real on restaurant `grade` later in this lecture.
`OrdinalEncoder` maps categories to numbers in an order **you specify explicitly** — never let it guess,
since alphabetical or first-seen order won't reliably match the order that actually matters.

In [ ]:
priority_encoder = __________(categories=[["Low", "Medium", "High"]])

sample_priorities = pd.DataFrame({"priority": ["Medium", "Low", "High", "Medium", "Low"]})
sample_priorities["priority_encoded"] = priority_encoder.fit_transform(sample_priorities[["priority"]])

sample_priorities

Low is now 0, Medium is 1, High is 2 — in that specific order, because we told the encoder that order
explicitly with `categories=[["Low", "Medium", "High"]]`. If you'd used one-hot encoding here instead,
the model would see three unrelated columns with no sense that Low and Medium are "closer" than Low and
High — technically usable, but throwing away real information you already have for free. Restaurant
`grade` later in this lecture is the same idea, on a real column.

---

## Part 3 — Scaling Numeric Features

In [ ]:
complaints_df[["hour_filed"]].__________()

`hour_filed` only ever runs 0-23. That's a narrow range on its own — but imagine combining it with
`resolution_time_hours`, which can run into the hundreds, as features side by side. That mismatch isn't
a problem for a human, but it matters a lot for a model:

- **k-NN (Week 8)**, which measures distance between rows — a feature with bigger raw numbers
  automatically contributes more to that distance, whether or not it's actually more informative
- **Linear/logistic regression (Week 7)**, where features on wildly different scales can slow down or
  destabilize the fitting process

**Scaling** puts every numeric feature onto a comparable range, so no feature dominates just because of
the units it happens to be measured in.

### Standardization (z-score scaling)

This is exactly the z-score idea from Lecture 5: subtract the mean, divide by the standard deviation.
After standardizing, every feature has a mean of 0 and a standard deviation of 1.

In [ ]:
numeric_cols = ["hour_filed"]

scaler = StandardScaler()
scaled_values = scaler.__________(complaints_df[numeric_cols])

scaled_df = pd.DataFrame(scaled_values, columns=[f"{c}_scaled" for c in numeric_cols])
print("Mean after scaling (should be ~0):", scaled_df.mean().round(4).tolist())
print("Std after scaling (should be ~1): ", scaled_df.std().round(4).tolist())
scaled_df.head()

`hour_filed` now sits centered at 0 with a spread of 1 — the same footing any other standardized
feature would be on, no matter its original units. That's the whole point: once you add a second numeric
feature later (as Part 5 does), both end up comparable instead of one silently dominating.

### Min-max scaling

An alternative: squeeze every value into a fixed range, usually 0 to 1, based on the minimum and
maximum observed.

In [ ]:
minmax_scaler = MinMaxScaler()
minmax_values = minmax_scaler.__________(complaints_df[numeric_cols])

minmax_df = pd.DataFrame(minmax_values, columns=[f"{c}_minmax" for c in numeric_cols])
print("Min after scaling (should be 0):", minmax_df.min().round(4).tolist())
print("Max after scaling (should be 1):", minmax_df.max().round(4).tolist())
minmax_df.head()

Min-max scaling is more sensitive to outliers than standardization — a single unusually extreme value
stretches the whole 0-1 range for everyone else. Standardization is the more common default for this
reason, but min-max is useful when you need values in a strictly bounded range.

One practical note worth remembering all semester: tree-based models (decision trees, random forests,
XGBoost — Week 10) don't need scaled features at all. Scaling only matters for distance-based and
gradient-based algorithms. It never hurts to scale, but it isn't always necessary.

---

## Part 4 — Data Leakage: The Most Important Rule in This Lecture

**Data leakage** happens when information from outside the training data — often, information from
your test set — accidentally influences how a model is built. It's one of the easiest ways to end up
with a model that looks great in your notebook and performs much worse in the real world, because your
evaluation was quietly cheating without you realizing it.

The rule that prevents almost all of it: **split your data first, then fit anything (scalers, encoders,
feature selection, the model itself) only on the training set.**

### The wrong way

Watch what happens if you scale *before* splitting.

In [ ]:
# WRONG -- fitting the scaler on the full dataset before splitting
leaky_scaler = StandardScaler()
leaky_scaled = leaky_scaler.__________(complaints_df[numeric_cols])

print("Mean learned by the LEAKY scaler (fit on everything):")
print(leaky_scaler.mean_.round(3))

That `mean_` was computed using **every row**, including the ones that should have been held out as a
test set. Any model trained on `leaky_scaled` afterward was implicitly given a hint about the test data's
distribution before it was ever supposed to see it. The model isn't reading test labels directly, but it's
no longer being evaluated on truly unseen data either — the test set's influence already leaked in
through the scaler.

### The right way

In [ ]:
# RIGHT -- split first, fit the scaler on the training data only
X = complaints_df[numeric_cols]
y = complaints_df["resolution_time_hours"]  # standing in for the real Week 7 target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=383)

correct_scaler = StandardScaler()
X_train_scaled = correct_scaler.__________(X_train)   # fit AND transform on train
X_test_scaled = correct_scaler.__________(X_test)         # transform ONLY, using train's fitted values

print("Mean learned by the CORRECT scaler (fit on training data only):")
print(correct_scaler.mean_.round(3))

Notice the pattern: `.fit_transform()` on the training data, `.transform()` only on the test data. The
test set is scaled *using the training set's own mean and standard deviation* — it never contributes to
computing those numbers itself. This is the single habit that prevents most leakage: fit once, on
training data, and reuse that same fitted object everywhere else.

### Seeing the difference directly

In [ ]:
comparison = pd.DataFrame({
    "feature": numeric_cols,
    "mean (leaky, full dataset)": leaky_scaler.mean_.round(3),
    "mean (correct, train only)": correct_scaler.__________.round(3),
})
comparison

The two means are close but not identical here — with a large, reasonably uniform dataset, the gap
can look small enough to shrug off. That's exactly what makes leakage dangerous: it rarely announces
itself. The same mistake gets much worse with a smaller dataset, a rare category that only appears in
your test split, or feature selection that peeks at test-set correlations before choosing which columns
to keep. The habit matters more than the size of the effect in any one example.

### A quick self-check for leakage

Ask yourself these before trusting any model's reported performance:

- Did I split my data **before** fitting any scaler, encoder, or feature selector?
- Did I call `.fit()` (or `.fit_transform()`) only on training data, everywhere in my pipeline?
- Does my test set only ever get `.transform()`, never `.fit()`?
- If I engineered a feature using some kind of aggregate (a group mean, a count, a ratio), did that
  aggregate get computed using training data only, not the full dataset?

Keep this checklist in mind for your capstone — you're required to discuss how you prevented leakage in
your own project, and this is exactly the list to walk through when you write that section.

---

## Part 5 — Bringing It Together with Pipelines

Encoding categoricals and scaling numerics separately works, but it's easy to make a mistake — forgetting
to refit something, or accidentally calling `.fit()` on the test set out of habit. scikit-learn's
`ColumnTransformer` bundles every preprocessing step into a single object, so `.fit()` and `.transform()`
always happen in the right place, on the right data. (Its cousin `Pipeline`, which chains preprocessing
*and* a model together, is what we'll use properly starting Week 11, once there's a real model to chain
it with — this is a first, practical preview of the preprocessing half.)

`resolution_time_hours` is what will become the regression target in Week 7, so it isn't a feature to
scale here — it's `y`, set aside like any other target. `hour_filed` (a genuine numeric feature) and
`complaint_type`/`borough` (nominal categoricals) are what get preprocessed.

In [ ]:
complaints_clean = complaints_df.dropna(subset=["resolution_time_hours"]).reset_index(drop=True)

X_311 = complaints_clean[["complaint_type", "borough", "hour_filed"]]
y_311 = complaints_clean["resolution_time_hours"]

X_311_train, X_311_test, y_311_train, y_311_test = train_test_split(
    X_311, y_311, test_size=__________, random_state=383
)

print("Training rows:", len(X_311_train))
print("Test rows:    ", len(X_311_test))

In [ ]:
preprocessor_311 = ColumnTransformer(transformers=[
    ("num", StandardScaler(), ["hour_filed"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["complaint_type", "borough"]),
])

X_311_train_ready = preprocessor_311.__________(X_311_train)
X_311_test_ready = preprocessor_311.__________(X_311_test)

print("Training features shape:", X_311_train_ready.shape)
print("Test features shape:    ", X_311_test_ready.shape)

One `preprocessor_311` object now handles both jobs. `fit_transform()` on `X_311_train` learns the
scaler's mean/std and the encoder's categories, all in one call, using training data only.
`transform()` on `X_311_test` reuses everything it learned. `X_311_train_ready` and `X_311_test_ready`
are fully numeric, properly scaled and encoded, fit only on the training split — exactly the inputs
Week 7's regression model will expect.

---

## Part 6 — Applied on Restaurant Inspections

Lecture 5 was 311 end to end. Assignment 1 was restaurant inspections end to end. Here's restaurant data
again — but this time, putting it through everything you just learned above: real one-hot encoding, a
real ordinal column, and a full leakage-safe pipeline, on messier, multi-featured data than 311 needed.

### Setup — NYC restaurant inspections

Same live-pull-with-fallback pattern as Lectures 4 and 5, at the level of one row per violation
citation — be careful not to call this "one row per inspection," since a single inspection can
contribute more than one row.

In [ ]:
try:
    raw_path = os.path.expanduser("~/shared/restaurant_inspections_snapshot.csv")
    inspections_df = pd.read_csv(raw_path)
    inspections_df["score"] = pd.to_numeric(inspections_df["score"], errors="coerce")
    inspections_df = inspections_df.dropna(subset=["score", "grade"]).reset_index(drop=True)

    # One row here is one violation citation, not one full inspection -- a single inspection can
    # contribute more than one row.
    inspections_df["is_critical"] = (inspections_df["critical_flag"] == "Critical").astype(int)
    live = True

except Exception:
    # Offline fallback, in case there is no internet connection in the room.
    rng = np.random.default_rng(383)
    n = 4000
    boroughs_list = ["MANHATTAN", "BROOKLYN", "QUEENS", "BRONX", "STATEN ISLAND"]
    cuisines_clean = ["American", "Chinese", "Italian", "Mexican", "Pizza", "Japanese"]

    is_critical = rng.integers(0, 2, size=n)
    grade = rng.choice(["A", "B", "C"], size=n, p=[0.6, 0.25, 0.15])
    score = rng.integers(0, 71, size=n)

    inspections_df = pd.DataFrame({
        "boro": rng.choice(boroughs_list, size=n, p=[0.22, 0.32, 0.26, 0.16, 0.04]),
        "cuisine_description": rng.choice(cuisines_clean, size=n),
        "score": score,
        "grade": grade,
        "critical_flag": np.where(is_critical == 1, "Critical", "Not Critical"),
        "is_critical": is_critical,
    })
    live = False

print(f"{'Shared snapshot' if live else 'Offline fallback'} data: {len(inspections_df):,} violation records")
inspections_df[["boro", "cuisine_description", "score", "grade", "is_critical"]].head()

In [ ]:
inspections_df.__________

Same story as Part 1, on a messier dataset: `boro`, `cuisine_description`, and `grade` are text.
`score` and `is_critical` are numeric. And this time there really is a genuine ordinal column —
`grade` — which 311 didn't give us.

### Nominal vs. ordinal, for real this time

- `boro` and `cuisine_description` are **nominal** — no natural order.
- `grade` is **ordinal** — A really is better than B is better than C.

### One-hot encoding `boro` and `cuisine_description`

Same tool as Part 2, just two nominal columns at once instead of one.

In [ ]:
cat_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
cat_encoded = cat_encoder.__________(inspections_df[["boro", "cuisine_description"]])

pd.DataFrame(cat_encoded, columns=cat_encoder.get_feature_names_out(["boro", "cuisine_description"])).head()

`boro` has 5 categories and `cuisine_description` has a dozen in this snapshot — check for yourself with
`inspections_df["cuisine_description"].nunique()`. One-hot encoding both together is exactly the same
call as encoding one; `OneHotEncoder` handles as many columns as you hand it.

### High cardinality: when one-hot encoding gets out of hand

In *this* snapshot, `cuisine_description`'s dozen categories happen to be fairly balanced — none
dominant, none especially rare. That won't always be true of your own data, so it's worth seeing the
fix on a case where it clearly matters, before you need it for real. Here's a small made-up example: a
cuisine list with a few common categories and several that show up only a handful of times.

In [ ]:
toy_cuisines = pd.Series(
    ["Italian"] * 200 + ["Chinese"] * 180 + ["Pizza"] * 150 + ["Mexican"] * 90
    + ["Thai"] * 4 + ["Ethiopian"] * 3 + ["Peruvian"] * 2 + ["Basque"] * 1
)
toy_counts = toy_cuisines.__________()
print(f"Distinct cuisines: {len(toy_counts)}")
toy_counts

In [ ]:
MIN_COUNT = 30

common_cuisines = toy_counts[toy_counts >= MIN_COUNT].index
toy_grouped = toy_cuisines.where(toy_cuisines.isin(common_cuisines), other=__________)

print(f"Categories before grouping: {toy_cuisines.nunique()}")
print(f"Categories after grouping:  {toy_grouped.nunique()}")
toy_grouped.value_counts()

One-hot encoding the untouched version would create one column per cuisine — Thai, Ethiopian,
Peruvian, and Basque would each get their own column representing a handful of rows or fewer. That's
not just wasteful: a category with 1-4 training examples gives a model almost nothing reliable to learn
from, and can make some models unstable. Grouping rare categories into `"Other"` before encoding fixes
that. `inspections_df["cuisine_description"]` doesn't need this today, but now you have the tool for
when a real dataset hands you a long tail like this toy example's.

### Ordinal encoding `grade`, for real

Part 2 previewed this on a made-up example. Here's the same tool on an actual ordinal column.

In [ ]:
grade_encoder = __________(categories=[["A", "B", "C"]])
inspections_df["grade_encoded"] = grade_encoder.fit_transform(inspections_df[["grade"]])

inspections_df[["grade", "grade_encoded"]].drop_duplicates().sort_values("grade_encoded")

A is 0, B is 1, C is 2 — the order we specified explicitly, exactly like the toy `priority` example in
Part 2, just on a real column this time.

### The full pipeline, split-first

Same discipline as Part 4: split before fitting anything. `score` stands in as the target here, the
way `resolution_time_hours` did for 311 — one `ColumnTransformer` this time handles a numeric passthrough,
a nominal one-hot, *and* an ordinal encoding, all in one fitted object.

In [ ]:
numeric_features = ["is_critical"]
nominal_features = ["boro", "cuisine_description"]
ordinal_features = ["grade"]

preprocessor_rest = ColumnTransformer(transformers=[
    ("num", StandardScaler(), numeric_features),
    ("nom", OneHotEncoder(handle_unknown="ignore"), nominal_features),
    ("ord", OrdinalEncoder(categories=[["A", "B", "C"]]), ordinal_features),
])

X_rest = inspections_df[numeric_features + nominal_features + ordinal_features]
y_rest = inspections_df["score"]

X_rest_train, X_rest_test, y_rest_train, y_rest_test = train_test_split(
    X_rest, y_rest, test_size=0.2, random_state=383
)

X_rest_train_ready = preprocessor_rest.__________(X_rest_train)
X_rest_test_ready = preprocessor_rest.__________(X_rest_test)

print("Training features shape:", X_rest_train_ready.shape)
print("Test features shape:    ", X_rest_test_ready.shape)

`X_rest_train_ready` and `X_rest_test_ready` are fully numeric, three different encoding strategies
bundled into one fitted object, fit only on the training split. `is_critical` is already 0/1, so
scaling it barely moves it — included here anyway for consistency, since it costs nothing and keeps
the pattern uniform if a future numeric feature isn't already so well-behaved. Nothing about the model
itself has been introduced yet; this is entirely the preparation step, on the second dataset you'll see
it applied to in Week 7.

---

**Exercises for this lecture** (Lab, Exit Ticket, Optional Challenge) live in a separate notebook:
`lect06_feature_engineering_exercise.ipynb`.

---

## Part 7 — Cheat Sheet

| Task | Code |
|---|---|
| One-hot encode (quick look) | `pd.get_dummies(df[["col"]])` |
| One-hot encode (for a pipeline) | `OneHotEncoder(handle_unknown="ignore")` |
| Ordinal encode, explicit order | `OrdinalEncoder(categories=[["A","B","C"]])` |
| Standardize (z-score) | `StandardScaler().fit_transform(X_train)` |
| Min-max scale | `MinMaxScaler().fit_transform(X_train)` |
| Fit on train, apply to test | `scaler.fit_transform(X_train)` then `scaler.transform(X_test)` |
| Bundle scaling + encoding | `ColumnTransformer([...])` |
| Group rare categories | `col.where(col.isin(common), other="Other")` |
| Split before any preprocessing | `train_test_split(X, y, test_size=..., random_state=...)` |

---

## Part 8 — Key Terms

- **Feature engineering**: preparing and transforming raw data into a form a model can use.
- **One-hot encoding**: representing a categorical column as multiple binary (0/1) columns, one per
  category.
- **Ordinal encoding**: mapping ordered categories to numbers that preserve their order.
- **Cardinality**: the number of distinct values in a categorical column; "high cardinality" means many
  distinct values, often with some appearing rarely.
- **Standardization (z-score scaling)**: rescaling a numeric feature to have mean 0 and standard
  deviation 1.
- **Min-max scaling**: rescaling a numeric feature to a fixed range, usually 0 to 1.
- **Data leakage**: information from outside the training data (often the test set) improperly
  influencing model training or evaluation, making performance look better than it really is.
- **`fit()` vs. `transform()`**: `fit()` learns parameters (a mean, a set of categories) from data;
  `transform()` applies previously learned parameters to data. The leakage rule: `fit()` only on
  training data, `transform()` everywhere else.
- **`ColumnTransformer`**: a scikit-learn tool that applies different preprocessing steps to different
  columns, bundled into a single fit/transform object.